# Εβδομάδα 5 — EDA για `Time` & `Amount` (Credit Card Fraud)

**Στόχος:** Να μελετήσουμε τις κατανομές των μεταβλητών `Time` και `Amount`, συνολικά και ανά κλάση (`Class=0` vs `Class=1`), να εντοπίσουμε ακραίες τιμές/ασυμμετρίες και να καταγράψουμε ευρήματα.

> **Σημείωση για `Time`:** Στο dataset του Kaggle, η στήλη `Time` είναι *δευτερόλεπτα από την πρώτη συναλλαγή* και όχι πραγματική ημερομηνία/ώρα. Για ερμηνεία, συχνά μετατρέπεται σε **ώρες ημέρας** με `hour = ((Time % 86400) // 3600)`.


In [34]:
# --- Ρυθμίσεις & Imports ---
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # για headless export σε PNG
import matplotlib.pyplot as plt

# Ρύθμισε τις διαδρομές σου με βάση το repo σου
DATA_PATH = Path("../../data/data_raw/creditcard.csv") 
REPORTS_DIR = Path("../../reports/")
FIGURES_DIR = Path("../../reports/figures/week5")
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.width", 160) # για να χωράει σε 160 χαρακτήρες
pd.set_option("display.max_columns", 50) # να δείχνει μέχρι 50 στήλες


In [35]:
# --- Φόρτωση δεδομένων & βασικοί έλεγχοι ---
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Λείπει το {DATA_PATH}. Βάλε το dataset στο data/data_raw/ και ξανατρέξε.")

df = pd.read_csv(DATA_PATH)
print("Σχήμα:", df.shape)
print("Στήλες:", df.columns.tolist()[:10], "...", df.columns.tolist()[-5:])

# Έλεγχος για απουσία/κενά
nulls = df.isna().sum().sort_values(ascending=False)
print("\nΚενές τιμές ανά στήλη (top 10):\n", nulls.head(10))

# Κατανομή κλάσης
class_counts = df["Class"].value_counts(dropna=False).rename(index={0:"non-fraud(0)", 1:"fraud(1)"}) 
fraud_rate = df["Class"].mean() 
print("\nΜετρήσεις κλάσεων:\n", class_counts)
print(f"Fraud rate: {fraud_rate:.5f} ({fraud_rate*100:.3f}%)")


Σχήμα: (284807, 31)
Στήλες: ['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9'] ... ['V26', 'V27', 'V28', 'Amount', 'Class']

Κενές τιμές ανά στήλη (top 10):
 Time    0
V1      0
V2      0
V3      0
V4      0
V5      0
V6      0
V7      0
V8      0
V9      0
dtype: int64

Μετρήσεις κλάσεων:
 Class
non-fraud(0)    284315
fraud(1)           492
Name: count, dtype: int64
Fraud rate: 0.00173 (0.173%)


In [37]:
# --- Περιγραφικά στατιστικά για Time & Amount ---
NUM_COLS = ["Time", "Amount"]
desc = df[NUM_COLS].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T # extra percentiles (1%, 5%, 95%, 99%) → χρήσιμο για να δεις outliers
desc["skew"] = df[NUM_COLS].skew() # ασυμμετρία
desc["kurtosis"] = df[NUM_COLS].kurtosis() # κύρτωση
desc_path = REPORTS_DIR / "week5_eda_time_amount/eda_time_amount_stats.csv"
desc.to_csv(desc_path, index=True)
desc


,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max,skew,kurtosis
Time,284807.0,94813.859575,47488.145955,0.0,2422.00,25297.60,54201.5,84692.0,139320.500,164143.4,170560.94,172792.00,-0.035568,-1.293530
Amount,284807.0,88.349619,250.120109,0.0,0.12,0.92,5.6,22.0,77.165,365.0,1017.97,25691.16,16.977724,845.092646


## 🕒 Time
- count = 284,807 → όλες οι γραμμές έχουν τιμή, δεν υπάρχουν κενά.
- mean = 94,813 sec (~26.3 ώρες) → ο μέσος χρόνος συναλλαγής είναι περίπου στη μέση του dataset.
- std = 47,488 sec (~13.2 ώρες) → αρκετά μεγάλη διασπορά, λογικό γιατί το dataset καλύπτει περίπου 2 μέρες.
- min = 0 → η πρώτη συναλλαγή.
- percentiles:
- 1% ≈ 2,422 sec (δηλ. ~40 λεπτά από την αρχή)
- 25% = 54,202 sec (~15 ώρες)
- 50% (median) = 84,692 sec (~23.5 ώρες)
- 75% = 139,320 sec (~38.7 ώρες)
- 99% = 170,560 sec (~47.4 ώρες)
- max = 172,792 sec (~48 ώρες) → dataset καλύπτει ακριβώς 2 μέρες.
- skew = -0.036 → πρακτικά συμμετρική κατανομή, δεν έχει έντονη ασυμμετρία.
- kurtosis = -1.29 → πιο “πλακουτσωτή” κατανομή από κανονική (δεν έχει βαριές ουρές).

📌 Συμπέρασμα: Το Time είναι απλά δείκτης σε δευτερόλεπτα, καλύπτει 2 μέρες. Δεν έχει πρόβλημα ασυμμετρίας, πιο πολύ χρησιμοποιείται για patterns στη διάρκεια της ημέρας (π.χ. πιο πολλά fraud σε συγκεκριμένες ώρες).

## 💶 Amount
- count = 284,807 → όλες οι συναλλαγές έχουν ποσό.
- mean = 88.35 € → μέσο ποσό ~88€.
- std = 250.12 € → πολύ μεγαλύτερο από τον μέσο όρο → δείχνει έντονη διασπορά (outliers).
- min = 0 → υπάρχουν συναλλαγές με ποσό 0 (π.χ. test/σφάλματα).
- percentiles:
- 1% ≈ 0.12 € (πολύ μικρά ποσά)
- 5% ≈ 0.92 €
- 25% = 5.6 €
- 50% (median) = 22 €
- 75% = 77.16 €
- 95% = 365 €
- 99% = 1,017.97 €
- max = 25,691 € → υπάρχουν συναλλαγές με υπερβολικά μεγάλα ποσά (outliers).
- skew = 16.98 → πολύ έντονα δεξιά skewed (πολλά μικρά amounts και λίγα τεράστια).
- kurtosis = 845.09 → πάρα πολύ υψηλή → σημαίνει ότι έχει “βαριές ουρές” (πολλά ακραία outliers).

📌 Συμπέρασμα: Το Amount έχει πάρα πολλά μικρά ποσά (π.χ. 1€, 5€, 20€) και πολύ λίγες συναλλαγές με μεγάλα ποσά (1000€, 5000€, 25000€). Αυτές οι λίγες ακραίες τιμές τραβάνε πολύ την κατανομή. Είναι κλασική περίπτωση όπου χρειάζεται logarithmic scale (log(Amount+1)) για καλύτερη ανάλυση και visualization.

### ✅ Άρα:
- Το Time είναι “καθαρό” feature, απλά δείχνει διάρκεια σε 2 μέρες.
- Το Amount είναι πολύ skewed με outliers → χρειάζεται log scaling ή robust scaling πριν το modeling.

In [38]:
# --- Κατανομές Time (δευτερόλεπτα και ώρες ημέρας) ---
# 1) Ιστόγραμμα Time (σε δευτερόλεπτα)
plt.figure(figsize=(8,5))
plt.hist(df["Time"].values, bins=100, density=False)
plt.title("Histogram of Time (seconds from first transaction)")
plt.xlabel("Time (seconds)")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "eda_time_hist_seconds.png", dpi=150)
plt.close()

# 2) Μετατροπή σε ώρα ημέρας (0–23)
hours = ((df["Time"] % 86400) // 3600).astype(int) # Το % 86400 → κρατάει το υπόλοιπο εντός της ημέρας (86400 sec = 24 ώρες). Το // 3600 → μετατρέπει σε ώρες ημέρας (0–23).0
plt.figure(figsize=(8,5))
plt.hist(hours, bins=np.arange(-0.5,24.5,1.0), rwidth=0.9)
plt.title("Histogram of Hour-of-Day derived from Time")
plt.xlabel("Hour of Day [0-23]")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "eda_time_hist_hour.png", dpi=150)
plt.close()


In [39]:
# --- Κατανομές Amount (απόλυτα & σε log1p) ---
# 1) Γραμμικό ιστόγραμμα
plt.figure(figsize=(8,5))
plt.hist(df["Amount"].values, bins=100, density=False)
plt.title("Histogram of Amount")
plt.xlabel("Amount")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "eda_amount_hist.png", dpi=150)
plt.close()
#Θα δεις τεράστια συγκέντρωση κοντά στο 0–100 €, και μερικά outliers που φτάνουν ~25.000 €.
#Επειδή τα μεγάλα amounts είναι σπάνια αλλά πολύ μεγάλα, η κατανομή είναι δεξιά-λοξή (right-skewed).

# 2) Λογαριθμική κλίμακα στον άξονα x
plt.figure(figsize=(8,5))
plt.hist(df["Amount"].values, bins=100, density=False)
plt.xscale("log") # “στριμώχνει” τα outliers και σου δείχνει καθαρότερα πώς κατανέμονται τα μικρά–μεσαία amounts
plt.title("Histogram of Amount (log x-scale)")
plt.xlabel("Amount (log scale)")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "eda_amount_hist_logx.png", dpi=150)
plt.close()

# 3) Ιστόγραμμα log1p(Amount) για μείωση ασυμμετρίας
amount_log1p = np.log1p(df["Amount"].clip(lower=0)) # log1p(x) = log(1+x), για να αποφύγουμε το log(0). Το clip(lower=0) είναι για να αποφύγουμε αρνητικά amounts (αν υπήρχαν)
plt.figure(figsize=(8,5))
plt.hist(amount_log1p.values, bins=100, density=False)
plt.title("Histogram of log1p(Amount)")
plt.xlabel("log1p(Amount)")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "eda_amount_hist_log1p.png", dpi=150)
plt.close()


In [40]:
# --- Σύγκριση κατανομών ανά κλάση (fraud vs non-fraud) ---
fraud = df[df["Class"]==1]
nonfraud = df[df["Class"]==0]

# A) Amount vs Class — επικαλυπτόμενα ιστογράμματα (normalized densities)
plt.figure(figsize=(8,5))
plt.hist(nonfraud["Amount"].values, bins=80, density=True, alpha=0.5, label="Class 0 (non-fraud)")
plt.hist(fraud["Amount"].values, bins=80, density=True, alpha=0.5, label="Class 1 (fraud)")
plt.title("Amount distribution by Class (density)")
plt.xlabel("Amount")
plt.ylabel("Density")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "eda_amount_by_class_density.png", dpi=150)
plt.close()

# B) log1p(Amount) για καλύτερη ορατότητα ουρών
plt.figure(figsize=(8,5))
plt.hist(np.log1p(nonfraud["Amount"].clip(lower=0)), bins=80, density=True, alpha=0.5, label="Class 0 (non-fraud)")
plt.hist(np.log1p(fraud["Amount"].clip(lower=0)), bins=80, density=True, alpha=0.5, label="Class 1 (fraud)")
plt.title("log1p(Amount) by Class (density)")
plt.xlabel("log1p(Amount)")
plt.ylabel("Density")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "eda_amount_log1p_by_class_density.png", dpi=150)
plt.close()

# C) Ώρα ημέρας vs Class
fraud_hours = ((fraud["Time"] % 86400) // 3600).astype(int)
nonfraud_hours = ((nonfraud["Time"] % 86400) // 3600).astype(int)

plt.figure(figsize=(8,5))
bins = np.arange(-0.5,24.5,1.0)
plt.hist(nonfraud_hours, bins=bins, density=True, alpha=0.5, label="Class 0 (non-fraud)", rwidth=0.9)
plt.hist(fraud_hours, bins=bins, density=True, alpha=0.5, label="Class 1 (fraud)", rwidth=0.9)
plt.title("Hour-of-Day distribution by Class (density)")
plt.xlabel("Hour of Day [0-23]")
plt.ylabel("Density")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "eda_hour_by_class_density.png", dpi=150)
plt.close()


In [41]:
# --- Αυτόματη παραγωγή σύντομων ευρημάτων σε Markdown ---
def qualit_skew(x):
    s = float(pd.Series(x).skew())
    if s > 2: return "πολύ έντονα δεξιά ασύμμετρη (heavy right-skew)"
    if s > 1: return "έντονα δεξιά ασύμμετρη"
    if s < -1: return "έντονα αριστερά ασύμμετρη"
    return "σχετικά συμμετρική/ήπια ασύμμετρη"

amount_skew_txt = qualit_skew(df["Amount"])
time_skew_txt = qualit_skew(df["Time"])

q_amt_99 = float(df["Amount"].quantile(0.99))
pct_amt_big = float((df["Amount"] > q_amt_99).mean() * 100.0)

# Top ώρες με βάση την πυκνότητα απάτης (προσοχή: ενδεικτικό)
hour_all = ((df["Time"] % 86400) // 3600).astype(int)
hour_tbl = pd.DataFrame({
    "hour": hour_all,
    "is_fraud": df["Class"].astype(int)
})
agg = hour_tbl.groupby("hour").agg(total=("is_fraud","count"), fraud=("is_fraud","sum"))
agg["fraud_rate_hour"] = agg["fraud"] / agg["total"]
top_hours = agg.sort_values("fraud_rate_hour", ascending=False).head(3)
top_hours_list = ", ".join([f"{int(h)} (rate={r:.3f})" for h,r in zip(top_hours.index, top_hours["fraud_rate_hour"])])

md = f"""
# Εβδομάδα 5 — Ευρήματα EDA (Time & Amount)

- **Fraud rate (συνολικό):** {df['Class'].mean():.4%}
- **Κατανομή Amount:** {amount_skew_txt}. Το άνω 1% των ποσών ξεκινά περίπου από **{q_amt_99:.2f}** (περίπου {pct_amt_big:.2f}% των συναλλαγών).
- **Κατανομή Time:** {time_skew_txt}. Θυμήσου ότι το `Time` είναι δευτερόλεπτα από την πρώτη συναλλαγή.
- **Ώρες με αυξημένο ποσοστό απάτης (ενδεικτικά):** {top_hours_list}

- **Οπτικοποιήσεις:** δες εικόνες στον φάκελο `reports/figures/week5`:
  - `eda_time_hist_seconds.png`, `eda_time_hist_hour.png`
  - `eda_amount_hist.png`, `eda_amount_hist_logx.png`, `eda_amount_hist_log1p.png`
  - `eda_amount_by_class_density.png`, `eda_amount_log1p_by_class_density.png`, `eda_hour_by_class_density.png`

## Παρατηρήσεις
1. Η κατανομή του `Amount` είναι συχνά δεξιά ασύμμετρη — η **log1p** μετασχημάτιση βελτιώνει την ορατότητα.
2. Οι κατανομές ανά κλάση δείχνουν αν οι **απάτες** συμβαίνουν σε **διαφορετικά εύρη ποσών** ή **συγκεκριμένες ώρες**.
3. Τα παραπάνω βοηθούν σε **επιλογή χαρακτηριστικών** (π.χ. χρήση `log1p(Amount)`, `hour`) και στην **επιλογή κατάλληλων metrics** (λόγω ανισορροπίας).
4. Τα patterns στις ώρες είναι **ενδεικτικά** και **όχι αιτιώδη** — απαιτούν περαιτέρω διερεύνηση σε επόμενες εβδομάδες.

"""

md_path = REPORTS_DIR / "05_eda_time_amount.md"
with open(md_path, "w", encoding="utf-8") as f:
    f.write(md)

print("Αποθηκεύτηκε:", md_path)
md


Αποθηκεύτηκε: ..\..\reports\05_eda_time_amount.md


'\n# Εβδομάδα 5 — Ευρήματα EDA (Time & Amount)\n\n- **Fraud rate (συνολικό):** 0.1727%\n- **Κατανομή Amount:** πολύ έντονα δεξιά ασύμμετρη (heavy right-skew). Το άνω 1% των ποσών ξεκινά περίπου από **1017.97** (περίπου 1.00% των συναλλαγών).\n- **Κατανομή Time:** σχετικά συμμετρική/ήπια ασύμμετρη. Θυμήσου ότι το `Time` είναι δευτερόλεπτα από την πρώτη συναλλαγή.\n- **Ώρες με αυξημένο ποσοστό απάτης (ενδεικτικά):** 2 (rate=0.017), 4 (rate=0.010), 3 (rate=0.005)\n\n- **Οπτικοποιήσεις:** δες εικόνες στον φάκελο `reports/figures/week5`:\n  - `eda_time_hist_seconds.png`, `eda_time_hist_hour.png`\n  - `eda_amount_hist.png`, `eda_amount_hist_logx.png`, `eda_amount_hist_log1p.png`\n  - `eda_amount_by_class_density.png`, `eda_amount_log1p_by_class_density.png`, `eda_hour_by_class_density.png`\n\n## Παρατηρήσεις\n1. Η κατανομή του `Amount` είναι συχνά δεξιά ασύμμετρη — η **log1p** μετασχημάτιση βελτιώνει την ορατότητα.\n2. Οι κατανομές ανά κλάση δείχνουν αν οι **απάτες** συμβαίνουν σε **διαφορετ

In [42]:
# --- Επόμενα βήματα (υπενθύμιση) ---
print("Πρότεινε commit:")
print("git add notebooks/week5/eda_time_amount.ipynb reports/* reports/figures/*.png")
print('git commit -m "week5: EDA για Time & Amount (notebook + στατιστικά + plots + report)"')
print("git push")


Πρότεινε commit:
git add notebooks/week5/eda_time_amount.ipynb reports/* reports/figures/*.png
git commit -m "week5: EDA για Time & Amount (notebook + στατιστικά + plots + report)"
git push
